In [0]:
# =============================================================================
# Notebook  : TEST_bronze_layer
# Location  : /HGI-Lakehouse-Pipeline/05_Tests/TEST_bronze_layer
# Purpose   : Validates all 8 bronze tables against client spec requirements.
#             Run after any bronze load to confirm DQ gates pass.
# Usage     : Set customer_id widget → Run All
# =============================================================================

In [0]:
# CELL 1
%run ../config/pipeline_config

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id",   "")
dbutils.widgets.text("source_system", "all")   # all | salesforce | bigquery

customer_id   = dbutils.widgets.get("customer_id").strip().lower()
source_filter = dbutils.widgets.get("source_system").strip().lower()

from datetime import datetime, timedelta
from pyspark.sql import functions as F

results = []

def test(name, fn):
    try:
        passed, detail = fn()
        results.append({"test": name, "passed": passed, "detail": detail})
        mark = "✅" if passed else "❌"
        print(f"  {mark} {'PASS' if passed else 'FAIL'} | {name}")
        if not passed:
            print(f"         Detail: {detail}")
    except Exception as e:
        results.append({"test": name, "passed": False, "detail": str(e)})
        print(f"  💥 ERROR | {name}: {e}")

def cnt(t):
    return spark.table(t).count()

def btbl(obj):
    return bronze_table(customer_id, obj)

print(f"=== Bronze Layer Test Suite ===")
print(f"  customer_id   : {customer_id}")
print(f"  source_filter : {source_filter}")
print(f"  Started       : {datetime.now()}")
print("=" * 65)

# ── SECTION 1: Infrastructure ────────────────────────────────────────────────
print("\n─── Infrastructure ───")

def test_bronze_schema_exists():
    schemas = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN {bronze_catalog}").collect()]
    exists = customer_id in schemas
    return exists, f"Schema '{bronze_catalog}.{customer_id}' {'found' if exists else 'NOT FOUND'}"
test("Bronze schema exists", test_bronze_schema_exists)

def test_all_bronze_tables_exist():
    missing = []
    for obj, tbl in BRONZE_TABLES.items():
        full = f"{bronze_catalog}.{customer_id}.{tbl}"
        try:
            spark.table(full)
        except Exception:
            missing.append(f"{tbl} (for {obj})")
    return len(missing) == 0, f"Missing tables: {missing}" if missing else "All 8 bronze tables exist"
test("All 8 bronze tables exist (incl. events_raw)", test_all_bronze_tables_exist)

def test_events_raw_table_name():
    """events_raw must be the BQ events table name — NOT crm_events"""
    expected = "events_raw"
    actual = BRONZE_TABLES.get("events")
    if actual != expected:
        return False, f"Expected '{expected}' but BRONZE_TABLES['events'] = '{actual}'"
    full = f"{bronze_catalog}.{customer_id}.{expected}"
    try:
        spark.table(full)
        return True, f"bronze.{customer_id}.events_raw exists"
    except Exception as e:
        return False, f"Table not found: {e}"
test("[KEY] BQ events table is named events_raw (not crm_events)", test_events_raw_table_name)

def test_watermark_table_exists():
    try:
        spark.table(f"{meta_catalog}.watermarks")
        return True, "ingestion_metadata.watermarks exists"
    except Exception as e:
        return False, str(e)
test("Watermark table exists", test_watermark_table_exists)

def test_delta_properties_set():
    issues = []
    for obj, tbl in BRONZE_TABLES.items():
        full = f"{bronze_catalog}.{customer_id}.{tbl}"
        try:
            props = {r["key"]: r["value"]
                     for r in spark.sql(f"SHOW TBLPROPERTIES {full}").collect()}
            if props.get("delta.autoOptimize.optimizeWrite") != "true":
                issues.append(f"{tbl}: optimizeWrite not set")
        except Exception as e:
            issues.append(f"{tbl}: {e}")
    return len(issues) == 0, "; ".join(issues) if issues else "Delta properties correct on all tables"
test("Delta table properties configured", test_delta_properties_set)

# ── SECTION 2: Data Presence ─────────────────────────────────────────────────
print("\n─── Data Presence ───")

def test_all_sf_tables_have_data():
    empty = []
    sf_objects = [o for o in BRONZE_TABLES if o != "events"]
    for obj in sf_objects:
        try:
            n = cnt(btbl(obj))
            if n == 0:
                empty.append(BRONZE_TABLES[obj])
        except Exception as e:
            empty.append(f"{BRONZE_TABLES[obj]}: {e}")
    return len(empty) == 0, f"Empty tables: {empty}" if empty else "All 7 SF bronze tables have data"
test("All 7 Salesforce bronze tables have data", test_all_sf_tables_have_data)

def test_events_raw_has_data():
    try:
        n = cnt(btbl("events"))
        return n > 0, f"events_raw: {n:,} rows"
    except Exception as e:
        return False, str(e)
test("events_raw has data", test_events_raw_has_data)

# ── SECTION 3: Composite ID Spec ─────────────────────────────────────────────
print("\n─── Composite ID Format (Spec §Step 3) ───")

def test_composite_id_format_accounts():
    df = spark.table(btbl("account"))
    bad = df.filter(~F.col("id").rlike(r"^salesforce_Account_Id_[A-Za-z0-9]{15,18}$")).count()
    return bad == 0, f"{bad} malformed composite IDs (expected: salesforce_Account_Id_...)"
test("Account composite IDs: salesforce_Account_Id_...", test_composite_id_format_accounts)

def test_composite_id_format_contacts():
    df = spark.table(btbl("contact"))
    bad = df.filter(~F.col("id").rlike(r"^salesforce_Contact_Id_[A-Za-z0-9]{15,18}$")).count()
    return bad == 0, f"{bad} malformed contact composite IDs"
test("Contact composite IDs: salesforce_Contact_Id_...", test_composite_id_format_contacts)

def test_composite_id_not_bare_sf_id():
    bad = spark.table(btbl("account")).filter(F.col("id") == F.col("source_key_value")).count()
    return bad == 0, f"{bad} records where id == source_key_value (missing prefix)"
test("Composite ID is not a bare Salesforce ID", test_composite_id_not_bare_sf_id)

def test_campaignmember_object_casing():
    """CampaignMember must NOT be 'Campaignmember' (capitalize() bug)"""
    df = spark.table(btbl("campaignmember"))
    wrong = df.filter(F.col("source_system_object") != "CampaignMember").count()
    if wrong > 0:
        sample = df.select("source_system_object").first()[0]
        return False, f"Expected 'CampaignMember', found '{sample}'"
    return True, "source_system_object = 'CampaignMember' (correct)"
test("[KEY] CampaignMember casing correct (not 'Campaignmember')", test_campaignmember_object_casing)

def test_bq_events_composite_id():
    df = spark.table(btbl("events"))
    bad = df.filter(~F.col("id").rlike(r"^bigquery_Event_event_id_")).count()
    return bad == 0, f"{bad} BQ events with wrong composite ID format"
test("BQ events composite ID: bigquery_Event_event_id_...", test_bq_events_composite_id)

# ── SECTION 4: Timestamp Spec ────────────────────────────────────────────────
print("\n─── Timestamps (Spec §Part 2) ───")

def test_no_corrupted_timestamps():
    """4.53e25 → should be NULL, not year-9999"""
    future_cutoff = datetime(2100, 1, 1)
    issues = []
    for obj in ["account", "contact", "opportunity"]:
        bad = spark.table(btbl(obj)).filter(
            F.col("data_timestamp") > F.lit(future_cutoff)
        ).count()
        if bad > 0:
            issues.append(f"{obj}: {bad} far-future timestamps")
    return len(issues) == 0, "; ".join(issues) if issues else "No 4.53e25 corruption"
test("[KEY] No corrupted timestamps (4.53e25 → NULL)", test_no_corrupted_timestamps)

def test_data_timestamp_not_null():
    issues = []
    for obj in ["account", "contact", "opportunity", "task"]:
        bad = spark.table(btbl(obj)).filter(F.col("data_timestamp").isNull()).count()
        if bad > 0:
            issues.append(f"{obj}: {bad} null")
    return len(issues) == 0, "; ".join(issues) if issues else "data_timestamp populated for all objects"
test("data_timestamp not null", test_data_timestamp_not_null)

# ── SECTION 5: Deduplication ─────────────────────────────────────────────────
print("\n─── Deduplication (Spec §Steps 1-2) ───")

def test_no_duplicate_4key():
    """No duplicate (source_system, source_system_object, source_key_name, source_key_value)"""
    issues = []
    for obj in ["account", "contact", "opportunity", "campaign"]:
        dupes = (
            spark.table(btbl(obj))
            .groupBy("source_system", "source_system_object",
                     "source_key_name", "source_key_value")
            .count().filter("count > 1").count()
        )
        if dupes > 0:
            issues.append(f"{obj}: {dupes} duplicate 4-key combos")
    return len(issues) == 0, "; ".join(issues) if issues else "No duplicate 4-column keys"
test("[KEY] No duplicate 4-column composite keys (intra-batch dedup)", test_no_duplicate_4key)

def test_record_hash_populated():
    issues = []
    for obj in ["account", "contact", "opportunity"]:
        df = spark.table(btbl(obj))
        if "record_hash" in df.columns:
            bad = df.filter(F.col("record_hash").isNull()).count()
            if bad > 0:
                issues.append(f"{obj}: {bad} null record_hash")
    return len(issues) == 0, "; ".join(issues) if issues else "record_hash populated (cross-batch detection ready)"
test("record_hash populated (xxhash64 fingerprint)", test_record_hash_populated)

# ── SECTION 6: Field-level Spec ──────────────────────────────────────────────
print("\n─── Field-Level Spec ───")

def test_contact_email_lowercase():
    df = spark.table(btbl("contact"))
    if "email" not in df.columns:
        return True, "email column not present yet"
    bad = df.filter(
        F.col("email").isNotNull() & (F.col("email") != F.lower(F.col("email")))
    ).count()
    return bad == 0, f"{bad} contacts with uppercase email"
test("Contact email is lowercase", test_contact_email_lowercase)

def test_user_email_lowercase():
    df = spark.table(btbl("user"))
    if "email" not in df.columns:
        return True, "email column not present"
    bad = df.filter(
        F.col("email").isNotNull() & (F.col("email") != F.lower(F.col("email")))
    ).count()
    return bad == 0, f"{bad} users with uppercase email"
test("User email is lowercase", test_user_email_lowercase)

def test_task_whoid_resolved():
    df = spark.table(btbl("task"))
    if "contact_source_key_value" not in df.columns:
        return True, "contact_source_key_value column not present"
    total    = cnt(btbl("task"))
    resolved = df.filter(F.col("contact_source_key_value").isNotNull()).count()
    pct = round(100 * resolved / max(total, 1), 1)
    return pct >= 50.0, f"{resolved}/{total} = {pct}% WhoId/WhatId resolved (threshold ≥50%)"
test("Task WhoId/WhatId resolved ≥50%", test_task_whoid_resolved)

def test_a_columns_are_string():
    issues = []
    for obj in ["account", "contact", "opportunity"]:
        for f in spark.table(btbl(obj)).schema:
            if f.name.startswith("a_") and str(f.dataType) != "StringType()":
                issues.append(f"{obj}.{f.name}: {f.dataType}")
    return len(issues) == 0, "; ".join(issues) if issues else "All a_ columns are STRING"
test("All a_ columns are STRING type", test_a_columns_are_string)

def test_a_columns_present():
    """At least 3 a_ columns expected on accounts"""
    cols = [c for c in spark.table(btbl("account")).columns if c.startswith("a_")]
    return len(cols) >= 3, f"{len(cols)} a_ columns found on accounts"
test("Dynamic a_ columns present", test_a_columns_present)

def test_no_cdf_columns_in_bronze():
    cdf_meta = {"_change_type", "_commit_version", "_commit_timestamp"}
    for obj in ["account", "contact"]:
        cols = {c.lower() for c in spark.table(btbl(obj)).columns}
        found = cdf_meta & cols
        if found:
            return False, f"{obj}: CDF columns leaked: {found}"
    return True, "No CDF metadata columns in bronze"
test("CDF internal columns not persisted in bronze", test_no_cdf_columns_in_bronze)

def test_no_null_ids():
    issues = []
    for obj, tbl in BRONZE_TABLES.items():
        full = f"{bronze_catalog}.{customer_id}.{tbl}"
        try:
            bad = spark.table(full).filter(
                F.col("id").isNull() | (F.trim(F.col("id")) == "")
            ).count()
            if bad > 0:
                issues.append(f"{tbl}: {bad} null/empty IDs")
        except Exception as e:
            issues.append(f"{tbl}: {e}")
    return len(issues) == 0, "; ".join(issues) if issues else "No null or empty composite IDs"
test("No null or empty composite IDs", test_no_null_ids)

def test_source_system_values():
    sf_wrong = spark.table(btbl("account")).filter(
        F.col("source_system") != "salesforce"
    ).count()
    bq_wrong = spark.table(btbl("events")).filter(
        F.col("source_system") != "bigquery"
    ).count()
    if sf_wrong > 0 or bq_wrong > 0:
        return False, f"SF wrong: {sf_wrong}, BQ wrong: {bq_wrong}"
    return True, "source_system values correct (salesforce / bigquery)"
test("source_system column values correct", test_source_system_values)

def test_core_columns_present():
    core = {"id", "tenant", "source_system", "source_system_object",
            "source_key_name", "source_key_value", "data_timestamp",
            "created_at", "updated_at", "status", "record_hash"}
    for obj, tbl in BRONZE_TABLES.items():
        full = f"{bronze_catalog}.{customer_id}.{tbl}"
        try:
            cols = {c.lower() for c in spark.table(full).columns}
            missing = core - cols
            if missing:
                return False, f"{tbl}: missing core columns: {missing}"
        except Exception as e:
            return False, f"{tbl}: {e}"
    return True, "All core tracking columns present in all 8 tables"
test("All core tracking columns present", test_core_columns_present)

# ── SECTION 7: Watermarks ────────────────────────────────────────────────────
print("\n─── Watermarks ───")

def test_watermarks_populated():
    try:
        n = spark.sql(f"""
            SELECT COUNT(*) FROM {meta_catalog}.watermarks
        """).collect()[0][0]
        return n > 0, f"{n} watermark rows found"
    except Exception as e:
        return False, str(e)
test("Watermarks populated", test_watermarks_populated)

def test_watermark_not_future():
    try:
        rows = spark.sql(f"""
            SELECT last_processed_timestamp
            FROM {meta_catalog}.watermarks
            LIMIT 1
        """).collect()
        if not rows:
            return False, "No watermark rows"
        wm = rows[0][0]
        return wm <= datetime.utcnow(), f"Watermark: {wm}"
    except Exception as e:
        return False, str(e)
test("Watermark is not in the future", test_watermark_not_future)

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
total  = len(results)
passed = sum(1 for r in results if r["passed"])
failed = total - passed

print(f"Results: {passed}/{total} passed, {failed} failed")
print(f"Finished: {datetime.now()}")

if failed > 0:
    print("\nFailing tests:")
    for r in results:
        if not r["passed"]:
            print(f"  ❌ {r['test']}")
            print(f"     {r['detail']}")
    raise Exception(f"Bronze test suite: {failed} test(s) failed.")
else:
    print("\n✅ All bronze tests passed!")
